In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🔍 Multi-Step Sales Research Workflow

## What We're Building

An automated sales research pipeline that performs:
1. **Company Lookup** — gather company info from a simulated database
2. **Contact Enrichment** — find decision-makers and their roles
3. **Outreach Draft** — write a personalized cold email
4. **QA Check** — review the draft for quality and compliance

## Architecture

```
Company Name
     │
     ▼
┌────────────┐    ┌─────────────┐    ┌──────────┐    ┌──────────┐
│ Company    │ ──▶│ Contact     │ ──▶│ Outreach │ ──▶│ QA       │
│ Lookup     │    │ Enrichment  │    │ Draft    │    │ Check    │
└────────────┘    └─────────────┘    └──────────┘    └──────────┘
```

## Key LangGraph Concept: Sequential Multi-Step Pipeline

A linear chain of nodes where each step enriches the state with
more information. Later nodes depend on earlier nodes' outputs.

## Stack
- **LangGraph** — sequential pipeline
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)
print("All imports successful!")

## Step 2 — Simulated Company Database & State

In [ ]:
class SalesState(TypedDict):
    target_company: str
    our_product: str           # What we're selling
    company_profile: str       # From lookup
    contacts: str              # Enriched contacts
    outreach_email: str        # Draft email
    qa_feedback: str           # QA review
    qa_passed: bool            # Did it pass QA?
    final_email: str           # Final approved email


# Simulated company database
COMPANY_DB = {
    "acme corp": {
        "name": "Acme Corp",
        "industry": "Manufacturing",
        "size": "5,000 employees",
        "revenue": "$800M annual",
        "hq": "Chicago, IL",
        "tech_stack": "SAP ERP, legacy on-prem servers, beginning cloud migration",
        "recent_news": "Announced $50M digital transformation initiative in Q1 2024. New CTO hired from AWS.",
        "pain_points": "Slow deployment cycles, manual QA processes, aging infrastructure",
        "contacts": [
            {"name": "David Park", "title": "CTO", "background": "Ex-AWS, 15 years in cloud infrastructure"},
            {"name": "Lisa Fernandez", "title": "VP Engineering", "background": "10 years at Acme, leads 200-person eng team"},
            {"name": "Tom Nguyen", "title": "Director of DevOps", "background": "Kubernetes certified, pushing for CI/CD adoption"},
        ]
    },
    "brightpath health": {
        "name": "BrightPath Health",
        "industry": "Healthcare / Healthtech",
        "size": "1,200 employees",
        "revenue": "$200M annual",
        "hq": "Boston, MA",
        "tech_stack": "AWS, React, Python/Django, PostgreSQL",
        "recent_news": "Series C funding of $75M. Expanding telehealth platform to 3 new states.",
        "pain_points": "HIPAA compliance complexity, scaling telehealth video, data analytics maturity",
        "contacts": [
            {"name": "Dr. Rachel Kim", "title": "CEO", "background": "Former physician, founded company in 2018"},
            {"name": "Marcus Johnson", "title": "CTO", "background": "Ex-Epic Systems, healthcare IT expert"},
            {"name": "Amy Liu", "title": "Head of Data", "background": "PhD in biostatistics, building analytics platform"},
        ]
    },
}

print(f"Company database: {list(COMPANY_DB.keys())}")

## Step 3 — Define Pipeline Nodes

In [ ]:
# --- Node 1: Company Lookup ---
def company_lookup(state: SalesState) -> dict:
    company_key = state["target_company"].lower()
    print(f"  🏢 Looking up: {state['target_company']}")

    if company_key in COMPANY_DB:
        info = COMPANY_DB[company_key]
        profile = (
            f"Company: {info['name']}\n"
            f"Industry: {info['industry']}\n"
            f"Size: {info['size']}\n"
            f"Revenue: {info['revenue']}\n"
            f"HQ: {info['hq']}\n"
            f"Tech Stack: {info['tech_stack']}\n"
            f"Recent News: {info['recent_news']}\n"
            f"Pain Points: {info['pain_points']}"
        )
    else:
        profile = f"Company '{state['target_company']}' not found in database. Limited info available."

    print(f"     ✓ Profile loaded ({len(profile)} chars)")
    return {"company_profile": profile}


# --- Node 2: Contact Enrichment ---
enrich_prompt = ChatPromptTemplate.from_template(
    """Based on this company profile and our product, identify the best
person to reach out to and explain why they're the right contact.

Company Profile:
{profile}

Available Contacts:
{contacts}

Our Product: {product}

For the best contact, explain:
1. Who to contact and their title
2. Why they're the right person (based on their background + our product)
3. Talking points that would resonate with them

Contact recommendation:"""
)


def contact_enrichment(state: SalesState) -> dict:
    print(f"  👥 Enriching contacts...")
    company_key = state["target_company"].lower()
    contacts_data = COMPANY_DB.get(company_key, {}).get("contacts", [])
    contacts_str = "\n".join(
        f"- {c['name']} ({c['title']}): {c['background']}" for c in contacts_data
    ) or "No contacts available"

    chain = enrich_prompt | llm | StrOutputParser()
    result = chain.invoke({
        "profile": state["company_profile"],
        "contacts": contacts_str,
        "product": state["our_product"]
    })
    print(f"     ✓ Best contact identified")
    return {"contacts": result}


# --- Node 3: Outreach Draft ---
email_prompt = ChatPromptTemplate.from_template(
    """Write a personalized cold outreach email for a sales rep.

Company Profile:
{profile}

Target Contact:
{contacts}

Our Product: {product}

Rules:
- Keep it under 150 words
- Reference their specific situation (recent news, pain points)
- Don't be pushy — offer value first
- Include a specific, low-commitment CTA (e.g., 15-min call, case study)
- Professional but human tone

Email (include Subject line):"""
)


def outreach_draft(state: SalesState) -> dict:
    print(f"  ✉️ Drafting outreach email...")
    chain = email_prompt | llm | StrOutputParser()
    email = chain.invoke({
        "profile": state["company_profile"],
        "contacts": state["contacts"],
        "product": state["our_product"]
    })
    print(f"     ✓ Email drafted ({len(email.split())} words)")
    return {"outreach_email": email}


# --- Node 4: QA Check ---
qa_prompt = ChatPromptTemplate.from_template(
    """Review this sales outreach email for quality and compliance.

Email:
{email}

Check for:
1. Personalization: Does it reference the prospect specifically?
2. Length: Is it under 150 words?
3. Tone: Professional but not robotic?
4. CTA: Clear and low-commitment?
5. Compliance: No false claims, no pressure tactics, no misleading language?
6. Spam triggers: Avoids words like "guaranteed", "limited time", "act now"?

Respond with:
VERDICT: PASS or FAIL
ISSUES: (list any issues, or "None")
SUGGESTIONS: (improvements if any)

Review:"""
)


def qa_check(state: SalesState) -> dict:
    print(f"  🔍 QA checking email...")
    chain = qa_prompt | llm | StrOutputParser()
    feedback = chain.invoke({"email": state["outreach_email"]})
    passed = "PASS" in feedback.upper().split("VERDICT")[-1][:20] if "VERDICT" in feedback.upper() else True
    print(f"     ✓ QA {'PASSED' if passed else 'FAILED'}")
    return {
        "qa_feedback": feedback,
        "qa_passed": passed,
        "final_email": state["outreach_email"] if passed else ""
    }


print("All 4 pipeline nodes defined")

## Step 4 — Build & Run the Pipeline

In [ ]:
# Build graph
workflow = StateGraph(SalesState)
workflow.add_node("company_lookup", company_lookup)
workflow.add_node("contact_enrichment", contact_enrichment)
workflow.add_node("outreach_draft", outreach_draft)
workflow.add_node("qa_check", qa_check)

workflow.set_entry_point("company_lookup")
workflow.add_edge("company_lookup", "contact_enrichment")
workflow.add_edge("contact_enrichment", "outreach_draft")
workflow.add_edge("outreach_draft", "qa_check")
workflow.add_edge("qa_check", END)

app = workflow.compile()
print("Sales research pipeline compiled")

In [ ]:
# Run for Acme Corp
print("="*60)
print("🚀 Running sales research for Acme Corp")
print("="*60)

result = app.invoke({
    "target_company": "Acme Corp",
    "our_product": "CloudDeploy — a CI/CD platform that reduces deployment time by 80% with zero-downtime releases and built-in testing",
    "company_profile": "",
    "contacts": "",
    "outreach_email": "",
    "qa_feedback": "",
    "qa_passed": False,
    "final_email": "",
})

print("\n" + "="*60)
print("📧 FINAL EMAIL:")
print("="*60)
print(result["outreach_email"])
print("\n📋 QA Feedback:")
print(result["qa_feedback"])

In [ ]:
# Run for BrightPath Health
print("\n" + "="*60)
print("🚀 Running sales research for BrightPath Health")
print("="*60)

result2 = app.invoke({
    "target_company": "BrightPath Health",
    "our_product": "DataShield — HIPAA-compliant data analytics platform with built-in PHI detection and audit trails",
    "company_profile": "",
    "contacts": "",
    "outreach_email": "",
    "qa_feedback": "",
    "qa_passed": False,
    "final_email": "",
})

print("\n" + "="*60)
print("📧 FINAL EMAIL:")
print("="*60)
print(result2["outreach_email"])
print("\n📋 QA Feedback:")
print(result2["qa_feedback"])

## 🧠 Key Concepts Recap

| Concept | What We Learned |
|---------|-----------------|
| **Sequential Pipeline** | Nodes execute in order, each enriching the state |
| **State Accumulation** | Later nodes access all prior nodes' outputs |
| **Tool + LLM Nodes** | `company_lookup` is pure data, others use LLM |
| **QA Gate** | Final node validates the output before delivery |

## 🔧 Extensions

- **Retry loop**: If QA fails, loop back to outreach_draft with feedback
- **Real data**: Integrate LinkedIn API, Clearbit, or Apollo for real enrichment
- **Batch processing**: Process 100 prospects in parallel
- **CRM integration**: Push approved emails to HubSpot/Salesforce